In [ ]:
import pandas as pd

sheet_id = "1NOmVVECrN8BJQ-ZAAjpWt0Hbmd2hNCg5"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)
df.head()


,Unnamed: 0,id,date,brand,region,clean_content,lang
0,0,9253,2025-03-04 13:44:28,Wana,Jijel,why does tethering fail,en
1,1,11064,2024-12-07 10:39:28,Djezzy,Ain Temouchent,working fine today thumbs up djezzy,en
2,2,7202,2024-12-04 14:28:28,Ooredoo,Mostaganem,why does tethering fail,en
3,3,16041,2025-01-30 2:52:29,Ooredoo,Ghardaia,test speed 02 mbps in ghardaya,en
4,4,18897,2025-10-13 1:14:29,Ooredoo,missing,test speed 02 mbps in eloued,en


In [ ]:
df.columns

Index(['Unnamed: 0', 'id', 'date', 'brand', 'region', 'clean_content', 'lang'], dtype='object')

In [ ]:
df.drop(columns = ["Unnamed: 0" , "id"])

,date,brand,region,clean_content,lang
0,2025-03-04 13:44:28,Wana,Jijel,why does tethering fail,en
1,2024-12-07 10:39:28,Djezzy,Ain Temouchent,working fine today thumbs up djezzy,en
2,2024-12-04 14:28:28,Ooredoo,Mostaganem,why does tethering fail,en
3,2025-01-30 2:52:29,Ooredoo,Ghardaia,test speed 02 mbps in ghardaya,en
4,2025-10-13 1:14:29,Ooredoo,missing,test speed 02 mbps in eloued,en
...,...,...,...,...,...
12057,2025-01-01 18:02:28,Djezzy,El Oued,why am i charged for sms i never sent httpssup...,en
12058,2025-01-01 3:20:28,Ooredoo,El Oued,no service since morning ooredoo,en
12059,2024-10-19 16:43:28,Mobilis,Biskra,app login fails with error code 403,en
12060,2025-07-25 10:30:29,Mobilis,Setif,app asks for permission every open,en


In [ ]:
df = df.rename(columns ={"clean_content":"content"})

# sentiment analysis (multiling)

In [ ]:
df.columns

Index(['Unnamed: 0', 'id', 'date', 'brand', 'region', 'content', 'lang'], dtype='object')

In [ ]:
!pip install -q transformers tqdm torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
import pandas as pd
import torch

# Make sure GPU is available
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

# Load model (fast tokenizer works fine here)
model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_model = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=16,
)



# Convert to list
texts = df['content'].astype(str).tolist()

# Run inference
results = []
for i in tqdm(range(0, len(texts), 32)):
    batch = texts[i:i+32]
    results.extend(sentiment_model(batch))

# Store results
df["sent_label"] = [r["label"] for r in results]
df["sent_score"] = [r["score"] for r in results]

df.head()


Using device: GPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Device set to use cuda:0


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]


  3%|▎         | 10/377 [00:01<00:51,  7.18it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset

100%|██████████| 377/377 [00:21<00:00, 17.46it/s]


,Unnamed: 0,id,date,brand,region,content,lang,sent_label,sent_score
0,0,9253,2025-03-04 13:44:28,Wana,Jijel,why does tethering fail,en,negative,0.941424
1,1,11064,2024-12-07 10:39:28,Djezzy,Ain Temouchent,working fine today thumbs up djezzy,en,positive,0.847947
2,2,7202,2024-12-04 14:28:28,Ooredoo,Mostaganem,why does tethering fail,en,negative,0.941424
3,3,16041,2025-01-30 2:52:29,Ooredoo,Ghardaia,test speed 02 mbps in ghardaya,en,neutral,0.585877
4,4,18897,2025-10-13 1:14:29,Ooredoo,missing,test speed 02 mbps in eloued,en,neutral,0.686481


In [ ]:
label_map = {
    "LABEL_0": "negative",
    "LABEL_1": "neutral",
    "LABEL_2": "positive"
}
df["sent_label"] = df["sent_label"].map(label_map)


In [ ]:
df["sent_label"].value_counts(normalize=True)


,proportion
sent_label,


# topic modeling (multiling)

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# multilingual MiniLM = light and good
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

docs = df["content"].astype(str).tolist()

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    verbose=True,
    min_topic_size=20
)

topics, probs = topic_model.fit_transform(docs)

df["topic_id"] = topics


2025-10-18 22:26:22,087 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/377 [00:00<?, ?it/s]

2025-10-18 22:26:28,192 - BERTopic - Embedding - Completed ✓
2025-10-18 22:26:28,193 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-18 22:27:22,459 - BERTopic - Dimensionality - Completed ✓
2025-10-18 22:27:22,461 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-18 22:27:22,989 - BERTopic - Cluster - Completed ✓
2025-10-18 22:27:22,996 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-18 22:27:23,167 - BERTopic - Representation - Completed ✓


In [ ]:
# get top words for each topic
topic_info = topic_model.get_topic_info()
topic_info.head()


,Topic,Count,Name,Representation,Representative_Docs
0,-1,827,-1_144p_only_streaming_entering,"[144p, only, streaming, entering, near, buildi...",[signal lost when entering building near boume...
1,0,227,0_speed_high_latency_data,"[speed, high, latency, data, low, ok, fine, bu...","[data speed fine but latency high in tipaza, d..."
2,1,218,1_fixed_it_team_impressed,"[fixed, it, team, impressed, fast, the, contac...","[the team fixed it fast impressed, the team fi..."
3,2,217,2_midcall_connection_mom_sad,"[midcall, connection, mom, sad, lost, with, co...","[lost connection midcall with mom sad, lost co..."
4,3,215,3_still_applied_promo_billing,"[still, applied, promo, billing, wrong, but, ,...","[promo applied but billing still wrong, promo ..."


In [ ]:
def get_topic_label(tid):
    if tid == -1:
        return "other"
    top_words = [word for word, _ in topic_model.get_topic(tid)[:3]]
    return ", ".join(top_words)

df["topic_name"] = df["topic_id"].apply(get_topic_label)


In [ ]:
df["topic_name"].value_counts().head(10)


,count
topic_name,
other,827
"speed, high, latency",227
"fixed, it, team",218
"midcall, connection, mom",217
"still, applied, promo",215
"crash, 500, error",212
"make, help, cant",212
"promos, bad, good",211
"recharge, billing, failed",207


In [ ]:
df[df["sent_label"]=="negative"]["topic_name"].value_counts().head(10)


,count
topic_name,


In [ ]:
# save df

In [ ]:
df.to_csv("feedback_with_sentiment_topics.csv", index=False)
print(" Saved with sentiment and topic columns.")


 Saved with sentiment and topic columns.
